# LLM 에이전트 교육 노트북 (vllm 버전)

## 학습 목표
이 노트북은 **vllm으로 서빙되는 사내 LLM 모델**을 활용해
AI 에이전트를 단계적으로 구축하는 방법을 가르칩니다.

> **vllm이란?**
> vllm은 오픈소스 LLM 추론 엔진으로 OpenAI API와 완전 호환됩니다.
> `base_url`만 사내 서버 주소로 변경하면 동일한 코드로 사내 모델을 사용할 수 있습니다.

---

## 5단계 학습 로드맵

| 단계 | 파일 | 핵심 개념 |
|------|------|---------|
| 1단계 | `1_llm_call.py` | 기본 LLM API 호출 |
| 2단계 | `2_tool_call.py` | 단일 도구 호출 (JSON 프로토콜) |
| 3단계 | `3_calculator_agent.py` | 에이전트 루프 (반복 도구 호출) |
| 4단계 | `4_coding_tools.py` | 파일 시스템 도구 구현 |
| 5단계 | `5_coding_agent.py` | 자율 코딩 에이전트 |

---

## 실행 모드 안내

첫 번째 코드 셀의 `USE_OPENAI_FOR_VALIDATION` 값으로 모드를 선택합니다:

| 값 | 모드 | 용도 |
|----|------|------|
| `True` | OpenAI API | 지금 바로 검증 (`.env`의 `OPENAI_API_KEY` 사용) |
| `False` | vllm 서버 | 사내 배포 환경 (`.env`의 `VLLM_BASE_URL`, `VLLM_MODEL_NAME` 사용) |

### `.env` 파일 설정

```
OPENAI_API_KEY=sk-...          # OpenAI API 키 (검증 모드)
VLLM_BASE_URL=http://localhost:8000/v1   # vllm 서버 주소 (vllm 모드)
VLLM_MODEL_NAME=your-model-name          # 서빙 중인 모델명 (vllm 모드)
```


In [1]:
# ============================================================
# 환경 설정: 모드 선택
# ============================================================
# USE_OPENAI_FOR_VALIDATION = True  → OpenAI API 직접 사용 (지금 바로 검증 가능)
# USE_OPENAI_FOR_VALIDATION = False → 사내 vllm 서버 사용 (실제 배포 환경)

USE_OPENAI_FOR_VALIDATION = True   # ← 이 값만 바꾸면 됩니다

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

if USE_OPENAI_FOR_VALIDATION:
    # ── 검증 모드: OpenAI API 직접 사용 ──────────────────────────
    # base_url을 지정하지 않으면 OpenAI 공식 서버에 연결됩니다
    VLLM_MODEL_NAME = "gpt-4.1"
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"[검증 모드] OpenAI API 사용")
    print(f"사용 모델: {VLLM_MODEL_NAME}")
else:
    # ── vllm 모드: 사내 서버 사용 ────────────────────────────────
    # .env 파일에 VLLM_BASE_URL, VLLM_MODEL_NAME 설정 필요
    VLLM_BASE_URL   = os.getenv("VLLM_BASE_URL",   "http://localhost:8000/v1")
    VLLM_MODEL_NAME = os.getenv("VLLM_MODEL_NAME", "your-model-name")
    API_KEY         = os.getenv("OPENAI_API_KEY",  "EMPTY")
    # vllm은 OpenAI API와 완전 호환 — base_url만 교체하면 됩니다
    client = OpenAI(base_url=VLLM_BASE_URL, api_key=API_KEY)
    print(f"[vllm 모드] 사내 서버 사용")
    print(f"서버 주소: {VLLM_BASE_URL}")
    print(f"사용 모델: {VLLM_MODEL_NAME}")


[검증 모드] OpenAI API 사용
사용 모델: gpt-4.1


---

## 1단계: 기본 LLM 호출 (`1_llm_call.py`)

### 개념

LLM(대형 언어 모델)과 통신하는 가장 기본적인 방법입니다.

```
사용자 질문 (prompt)
     ↓
OpenAI 호환 API (chat.completions.create)
     ↓
LLM 응답 텍스트
```

### 핵심 구조

- **`messages` 리스트**: 대화 기록을 담는 표준 형식 (`role`, `content`)
  - `role: "user"` → 사용자 발화
  - `role: "assistant"` → LLM 응답 (다음 단계에서 사용)
- **`client.chat.completions.create()`**: API 호출 메서드
- **`choices[0].message.content`**: 응답 텍스트 추출 경로


In [2]:
# ============================================================
# 1단계: 기본 LLM 호출 함수 정의
# ============================================================

def llm_call(prompt: str) -> str:
    """LLM에 단순 질문을 보내고 텍스트 응답을 반환합니다."""
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=VLLM_MODEL_NAME,  # .env의 VLLM_MODEL_NAME 사용
        messages=messages,
    )
    return response.choices[0].message.content

print("llm_call 함수 정의 완료")


llm_call 함수 정의 완료


In [3]:
# 실행: 기본 LLM 호출 테스트
prompt = "한국의 수도는 어디일까?"
response = llm_call(prompt)

print("질문:", prompt)
print("LLM 답변:", response)


질문: 한국의 수도는 어디일까?
LLM 답변: 한국의 수도는 **서울**입니다.  
서울특별시는 대한민국의 정치, 경제, 문화, 교육의 중심지이며, 한반도의 북서부에 위치해 있습니다.


---

## 2단계: 단일 도구 호출 (`2_tool_call.py`)

### 개념

LLM에게 **"어떤 함수를 어떤 인자로 호출할지"** JSON 형식으로 알려달라고 요청합니다.
LLM은 실제 계산을 수행하지 않고, 도구 호출 명세만 생성합니다.

```
사용자 질문 ("1 더하기 3은?")
     ↓
LLM → JSON 응답: {"tool_name": "add", "tool_parameters": {"a": 1, "b": 3}}
     ↓
파이썬 코드가 JSON을 파싱해서 add(1, 3) 직접 실행
     ↓
결과: 4
```

### 왜 JSON인가?

LLM은 텍스트를 생성하지만, 우리는 **구조화된 데이터**가 필요합니다.
JSON 형식을 강제하면 파싱이 쉬워집니다.

### 핵심 패턴

- **`TOOLS_MAP`**: 도구 이름 → 실제 파이썬 함수 매핑 딕셔너리
- **프롬프트 엔지니어링**: LLM에게 JSON 형식 출력을 강제하는 지시문


In [4]:
# ============================================================
# 2단계: 도구 정의 및 단일 도구 호출
# ============================================================

def add(a, b):
    """두 수를 더합니다."""
    return a + b

# 도구 이름 → 실제 파이썬 함수 매핑
TOOLS_MAP_STEP2 = {
    "add": add,
}

# LLM에게 JSON 형식으로만 응답하도록 요청하는 프롬프트 템플릿
# {question} 자리에 실제 질문이 들어갑니다
TOOL_CALL_PROMPT = '''
다음의 도구만 사용해서 문제를 해결하세요.

도구 목록:
- add: 두 수를 더합니다. (매개변수: a: number, b: number)

문제:
{question}

반드시 아래 JSON 형식으로만 답하세요. 자연어 설명 없이 도구명과 파라미터만 출력하세요.

{{
  "tool_name": "도구이름",
  "tool_parameters": {{"a": (첫 번째 숫자), "b": (두 번째 숫자)}}
}}
'''

print("2단계 도구 및 프롬프트 정의 완료")


2단계 도구 및 프롬프트 정의 완료


In [5]:
# 실행: LLM이 어떤 도구를 어떤 인자로 쓸지 결정하게 하기
question = "1 더하기 3은 뭐야 (반드시 도구 사용해야 함)"

prompt = TOOL_CALL_PROMPT.format(question=question)
llm_response = llm_call(prompt)
print("LLM 응답 (JSON):", llm_response)

# LLM이 반환한 JSON 파싱
result_json = json.loads(llm_response)
tool_name = result_json["tool_name"]
params    = result_json["tool_parameters"]

# 도구 이름으로 실제 파이썬 함수 호출
if tool_name in TOOLS_MAP_STEP2:
    result = TOOLS_MAP_STEP2[tool_name](**params)
    print(f"\n실행: {tool_name}({params}) = {result}")
else:
    print("알 수 없는 도구:", tool_name)


LLM 응답 (JSON): {
  "tool_name": "add",
  "tool_parameters": {"a": 1, "b": 3}
}

실행: add({'a': 1, 'b': 3}) = 4


---

## 3단계: 계산기 에이전트 (`3_calculator_agent.py`)

### 개념

복잡한 수식은 한 번의 도구 호출로 풀 수 없습니다.
**에이전트 루프**를 통해 LLM이 여러 단계로 나눠 반복 호출합니다.

```
[루프 시작]
  LLM에게 현재 이력 + 질문 전달
       ↓
  LLM: final_answer == "no"  → 도구 호출 → 결과 이력에 추가 → 반복
  LLM: final_answer == "yes" → 루프 종료, 최종 답 반환
[루프 종료]
```

### 핵심 패턴

- **`history` 문자열**: 이전 도구 호출과 결과를 누적해서 LLM에게 전달
  → LLM이 "지금까지 어디까지 계산했는지"를 알 수 있게 함
- **`final_answer` 필드**: LLM 스스로 "계산이 끝났다"를 판단하게 하는 신호
- **`max_iterations`**: 무한 루프 방지 안전장치

### 에이전트 실행 예시: `(1 + 1) * 2 - 1`

| 반복 | 도구 | 입력 | 결과 |
|------|------|------|------|
| 1 | add | a=1, b=1 | 2 |
| 2 | multiply | a=2, b=2 | 4 |
| 3 | subtract | a=4, b=1 | 3 |
| 4 | (종료) | final_answer=yes | "답은 3입니다" |


In [6]:
# ============================================================
# 3단계: 계산기 에이전트 (반복 도구 호출 루프)
# ============================================================

# add는 2단계에서 이미 정의했으므로 재사용
def multiply(a, b): return a * b
def subtract(a, b): return a - b

TOOLS_MAP_CALC = {
    "add":      add,        # 2단계에서 정의
    "multiply": multiply,
    "subtract": subtract,
}

CALC_AGENT_PROMPT = '''
아래의 도구들만 사용하여 문제를 단계별로 해결하세요.

도구 목록:
- add: 두 수를 더합니다. (매개변수: a: number, b: number)
- multiply: 두 수를 곱합니다. (매개변수: a: number, b: number)
- subtract: 첫 번째 숫자에서 두 번째 숫자를 뺍니다. (매개변수: a: number, b: number)

각 단계에서 반드시 아래 JSON 형식으로 하나의 도구만 호출하세요:

아직 최종 답이 아니면:
{{
  "final_answer": "no",
  "tool_name": "도구이름",
  "tool_parameters": {{ ...매개변수... (항상 숫자형!) }}
}}

최종 답이면:
{{
  "final_answer": "yes",
  "final_response": "질문에 대한 최종 응답을 자연스러운 언어로 표현"
}}

반드시 위의 JSON만 출력하세요. 자연어 설명은 쓰지 마세요.
tool_parameters 값은 반드시 숫자여야 합니다. (예: "a": 3, "b": 4.5)

이전 도구 호출 및 결과 이력:
{history}

문제 또는 목표:
{question}
'''

def run_calculator_agent(user_question, max_iterations=10):
    """에이전트 루프: LLM이 스스로 도구를 선택하며 반복 계산합니다."""
    history = ""
    for iteration in range(max_iterations):
        print(f"\n--- 반복 {iteration + 1} ---")

        # 현재까지의 이력을 포함해 LLM 호출
        prompt = CALC_AGENT_PROMPT.format(
            history=history or "(없음)",
            question=user_question
        )
        llm_response = llm_call(prompt)
        print(f"[LLM 응답]\n{llm_response}")

        tool_call = json.loads(llm_response)

        # LLM이 "끝났다"고 판단하면 루프 종료
        if tool_call.get("final_answer") == "yes":
            final_response = tool_call.get("final_response", "")
            print(f"\n최종 답: {final_response}")
            return final_response

        # 도구 실행 후 결과를 이력에 추가
        tool_name   = tool_call["tool_name"]
        tool_params = tool_call.get("tool_parameters", {})

        if tool_name in TOOLS_MAP_CALC:
            result = TOOLS_MAP_CALC[tool_name](**tool_params)
            history += f"[{tool_name}] {tool_params} → result={result}\n"
        else:
            print(f"알 수 없는 도구: {tool_name}. 루프를 중단합니다.")
            break

    print("최대 반복 횟수 초과")

print("3단계 계산기 에이전트 정의 완료")


3단계 계산기 에이전트 정의 완료


In [7]:
# 실행: (1 + 1) * 2 - 1 계산 (3번의 도구 호출 필요)
user_question = "다음 식을 계산해 주세요: (1 + 1) * 2 - 1. 반드시 각 연산마다 도구만 사용하세요."
run_calculator_agent(user_question)



--- 반복 1 ---
[LLM 응답]
{
  "final_answer": "no",
  "tool_name": "add",
  "tool_parameters": { "a": 1, "b": 1 }
}

--- 반복 2 ---
[LLM 응답]
{
  "final_answer": "no",
  "tool_name": "multiply",
  "tool_parameters": { "a": 2, "b": 2 }
}

--- 반복 3 ---
[LLM 응답]
{
  "final_answer": "no",
  "tool_name": "subtract",
  "tool_parameters": { "a": 4, "b": 1 }
}

--- 반복 4 ---
[LLM 응답]
{
  "final_answer": "yes",
  "final_response": "주어진 식 (1 + 1) * 2 - 1의 결과는 3입니다."
}

최종 답: 주어진 식 (1 + 1) * 2 - 1의 결과는 3입니다.


'주어진 식 (1 + 1) * 2 - 1의 결과는 3입니다.'

---

## 4단계: 파일 시스템 도구 (`4_coding_tools.py`)

### 개념

코딩 에이전트가 파일을 다루기 위한 3가지 핵심 도구를 구현합니다.
LLM이 직접 파일을 읽고 수정하려면 이런 인터페이스가 필요합니다.

| 도구 | 역할 | 주요 매개변수 |
|------|------|-------------|
| `list_files` | 폴더 내 파일/폴더 목록 조회 | `directory` |
| `read_file` | 파일 내용 읽기 | `file_path` |
| `edit_file` | 텍스트 교체 또는 파일 생성 | `file_path`, `old_text`, `new_text` |

### `edit_file` 동작 방식

```
파일이 존재하고 old_text가 주어진 경우 → 텍스트 교체 (replace)
파일이 없는 경우                        → 새 파일 생성 (create)
```

> **설계 원칙**: LLM은 도구의 입출력 형식만 알면 됩니다.
> 내부 구현(파일 I/O, 에러 처리)은 파이썬이 담당합니다.


In [8]:
# ============================================================
# 4단계: 파일 시스템 도구 구현
# ============================================================

def list_files(directory: str = ".") -> str:
    """지정한 폴더의 파일과 하위 폴더 목록을 반환합니다."""
    try:
        if not os.path.exists(directory):
            return f"존재하지 않는 경로입니다: {directory}"
        items = []
        for item in sorted(os.listdir(directory)):
            item_path = os.path.join(directory, item)
            tag = "[폴더]" if os.path.isdir(item_path) else "[파일]"
            items.append(f"{tag} {item}")
        if not items:
            return f"비어있는 폴더: {directory}"
        return f"{directory}의 내용:\n" + "\n".join(items)
    except Exception as e:
        return f"오류: {e}"

def read_file(file_path: str) -> str:
    """파일의 내용을 UTF-8로 읽어 반환합니다."""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    except Exception:
        return ""

def edit_file(file_path: str, old_text: str = "", new_text: str = "") -> str:
    """파일에서 old_text를 new_text로 교체합니다. 파일이 없으면 새로 생성합니다."""
    try:
        if os.path.exists(file_path) and old_text:
            content = read_file(file_path)
            if old_text not in content:
                return f"파일에서 텍스트를 찾을 수 없습니다: '{old_text}'"
            content = content.replace(old_text, new_text)
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(content)
            return f"수정 완료: {file_path}"
        else:
            dir_name = os.path.dirname(file_path)
            if dir_name:
                os.makedirs(dir_name, exist_ok=True)
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_text)
            return f"파일 생성 완료: {file_path}"
    except Exception as e:
        return f"오류: {e}"

print("4단계 파일 도구 정의 완료")


4단계 파일 도구 정의 완료


In [9]:
# 실행: 각 파일 도구 직접 테스트

print("=== list_files: test_files 폴더 목록 ===")
print(list_files("test_files"))

print("\n=== read_file: test_files/a.py 내용 ===")
print(read_file("test_files/a.py"))

print("\n=== edit_file: 새 파일 생성 테스트 ===")
result = edit_file(
    "test_files/new_test.py",
    new_text="# 에이전트가 생성한 테스트 파일\nprint('새로 만든 파일')\n"
)
print(result)
print("\n생성된 파일 내용:")
print(read_file("test_files/new_test.py"))


=== list_files: test_files 폴더 목록 ===
test_files의 내용:
[파일] a.py
[파일] b.py
[파일] c.py

=== read_file: test_files/a.py 내용 ===
import math

def foo_renamed_renamed():
    print("hello from a")


=== edit_file: 새 파일 생성 테스트 ===
파일 생성 완료: test_files/new_test.py

생성된 파일 내용:
# 에이전트가 생성한 테스트 파일
print('새로 만든 파일')



---

## 5단계: 자율 코딩 에이전트 (`5_coding_agent.py`)

### 개념

**3단계의 에이전트 루프** + **4단계의 파일 도구**를 결합합니다.
LLM이 파일을 직접 탐색하고 수정하는 **자율 코딩 에이전트**입니다.

```
[에이전트 시작]
  LLM: "어떤 파일이 있는지 먼저 확인해야겠어"  → list_files 호출
  LLM: "a.py 내용을 확인해야겠어"              → read_file 호출
  LLM: "'def foo'를 찾았어, 바꿔야겠어"         → edit_file 호출
  LLM: "모든 파일 수정 완료"                    → final_answer: "yes"
[에이전트 종료]
```

### 전체 아키텍처 요약

```
사용자 요청
     ↓
run_coding_agent()    ← 에이전트 루프 (3단계 패턴 재사용)
     ↓
llm_call()            ← vllm API 호출 (1단계 함수)
     ↓
JSON 파싱 → 도구 선택 → 파일 도구 실행 (4단계 도구)
     ↓
history에 결과 추가 → 다음 반복
     ↓
final_answer == "yes" → 종료
```

### 이 패턴의 확장 가능성

동일한 루프 구조에 도구만 교체하면 다양한 에이전트를 만들 수 있습니다:
- 웹 검색 도구 → 리서치 에이전트
- DB 쿼리 도구 → 데이터 분석 에이전트
- API 호출 도구 → 업무 자동화 에이전트


In [10]:
# ============================================================
# 5단계: 자율 코딩 에이전트
# ============================================================

# list_files, read_file, edit_file은 4단계에서 이미 정의
CODING_TOOLS_MAP = {
    "list_files": list_files,
    "read_file":  read_file,
    "edit_file":  edit_file,
}

CODING_AGENT_PROMPT = '''
아래 도구만 사용해서 문제를 단계별로 해결하세요.

도구 목록:
- list_files: 지정한 폴더의 파일/폴더 이름 목록을 반환합니다. (매개변수: directory)
- read_file: 파일 경로에 해당하는 텍스트 파일의 내용을 반환합니다. (매개변수: file_path)
- edit_file: 파일에서 문자열을 교체하거나 새 파일을 생성합니다. (매개변수: file_path, old_text, new_text)

각 단계에서 반드시 아래 JSON 형식으로 하나의 도구만 호출하세요:

아직 최종 답이 아니면:
{{
  "final_answer": "no",
  "tool_name": "도구이름",
  "tool_parameters": {{ ...매개변수... }}
}}

최종 답이면:
{{
  "final_answer": "yes",
  "final_response": "질문에 대한 최종 응답을 자연스러운 언어로 표현"
}}

반드시 위의 JSON만 출력하세요. 자연어 설명이나 기타 텍스트는 쓰지 마세요.

이전 도구 호출 및 결과 이력:
{history}

문제 또는 목표:
{question}
'''

def run_coding_agent(user_question, max_iterations=10):
    """파일 시스템 도구를 이용해 자율적으로 코드를 탐색하고 수정하는 에이전트."""
    history = ""
    for iteration in range(max_iterations):
        print(f"\n--- 반복 {iteration + 1} ---")

        prompt = CODING_AGENT_PROMPT.format(
            history=history or "(없음)",
            question=user_question
        )
        llm_response = llm_call(prompt)
        print(f"[LLM 응답]\n{llm_response}")

        tool_call = json.loads(llm_response)

        if tool_call.get("final_answer") == "yes":
            final_response = tool_call.get("final_response", "")
            print(f"\n최종 답: {final_response}")
            return final_response

        tool_name   = tool_call["tool_name"]
        tool_params = tool_call.get("tool_parameters", {})

        if tool_name in CODING_TOOLS_MAP:
            result = CODING_TOOLS_MAP[tool_name](**tool_params)
            # 긴 결과는 300자까지만 이력에 기록
            result_preview = result[:300] + "..." if len(result) > 300 else result
            print(f"[도구 실행] {tool_name}({tool_params})")
            print(f"[결과] {result_preview}")
            history += f"[{tool_name}] {tool_params} → result={result}\n"
        else:
            print(f"알 수 없는 도구: {tool_name}. 루프를 중단합니다.")
            break

    print("최대 반복 횟수 초과")

print("5단계 코딩 에이전트 정의 완료")


5단계 코딩 에이전트 정의 완료


In [11]:
# 실행: test_files 폴더의 모든 파일에서 함수명 자동 변경
# 에이전트가 자동으로: 파일 목록 확인 → 각 파일 읽기 → 해당 문자열 수정
user_question = "test_files 폴더 내 모든 파일에서 'def foo'를 'def foo_renamed'로 바꿔줘"
run_coding_agent(user_question)



--- 반복 1 ---
[LLM 응답]
{
  "final_answer": "no",
  "tool_name": "list_files",
  "tool_parameters": {
    "directory": "test_files"
  }
}
[도구 실행] list_files({'directory': 'test_files'})
[결과] test_files의 내용:
[파일] a.py
[파일] b.py
[파일] c.py
[파일] new_test.py

--- 반복 2 ---
[LLM 응답]
{
  "final_answer": "no",
  "tool_name": "read_file",
  "tool_parameters": {
    "file_path": "test_files/a.py"
  }
}
[도구 실행] read_file({'file_path': 'test_files/a.py'})
[결과] import math

def foo_renamed_renamed():
    print("hello from a")


--- 반복 3 ---
[LLM 응답]
{
  "final_answer": "no",
  "tool_name": "read_file",
  "tool_parameters": { "file_path": "test_files/b.py" }
}
[도구 실행] read_file({'file_path': 'test_files/b.py'})
[결과] import os
import math as m as m

def bar():
    print("hello from b")


--- 반복 4 ---
[LLM 응답]
{
  "final_answer": "no",
  "tool_name": "read_file",
  "tool_parameters": {
    "file_path": "test_files/c.py"
  }
}
[도구 실행] read_file({'file_path': 'test_files/c.py'})
[결과] # No math import here

"'test_files' 폴더 내의 모든 파일에서 'def foo'를 'def foo_renamed'로 변경했습니다. 'a.py'에만 해당 코드가 있었으며, 나머지 파일에는 없었습니다."